In [2]:
users_interests = [
    ["Hadoop", "Big Data", "HBase", "Java", "Spark", "Storm", "Cassandra"],
    ["NoSQL", "MongoDB", "Cassandra", "HBase", "Postgres"],
    ["Python", "scikit-learn", "scipy", "numpy", "statsmodels", "pandas"],
    ["R", "Python", "statistics", "regression", "probability"],
    ["machine learning", "regression", "decision trees", "libsvm"],
    ["Python", "R", "Java", "C++", "Haskell", "programming languages"],
    ["statistics", "probability", "mathematics", "theory"],
    ["machine learning", "scikit-learn", "Mahout", "neural networks"],
    ["neural networks", "deep learning", "Big Data", "artificial intelligence"],
    ["Hadoop", "Java", "MapReduce", "Big Data"],
    ["statistics", "R", "statsmodels"],
    ["C++", "deep learning", "artificial intelligence", "probability"],
    ["pandas", "R", "Python"],
    ["databases", "HBase", "Postgres", "MySQL", "MongoDB"],
    ["libsvm", "regression", "support vector machines"]
]

In [3]:
from collections import Counter


popular_interests = Counter(interest 
                            for users_interest in users_interests for interest in users_interest)
popular_interests.most_common()[0]

('Python', 4)

In [4]:
from typing import List, Tuple

def most_popular_new_interest(popular_interests: Counter, users_interest:List[str], max_results: int = 5) ->List[Tuple[str, int]]:
    'sugeruje 5 najpolrajniejszych pozycji ktorych nie ma w swojej liscie zainteresowan'
    suggestions = [interest for interest in popular_interests.most_common() if interest[0] not in users_interest]
    return suggestions[:max_results]

most_popular_new_interest(popular_interests, ["NoSQL", "MongoDB", "Cassandra", "HBase", "Postgres"])

[('Python', 4), ('R', 4), ('Big Data', 3), ('Java', 3), ('statistics', 3)]

In [5]:
#user_based collaborative filtering

unique_interests = sorted({interest for user_interest in users_interests for interest in user_interest}) # sortownie alfabeycznie
unique_interests[:5] 

['Big Data', 'C++', 'Cassandra', 'HBase', 'Hadoop']

In [6]:
import numpy as np

def cosine_sim(a, b):
    return (np.dot(a,b)) / (np.linalg.norm(a) * np.linalg.norm(b))

def make_user_interest_vector(users_interest:List[str]) -> List[int]:
    'wektor na podstawie unique_interests warotsci 1 jezeli zainteresowania sie pokrywaja else 0'
    return [1 if interest in users_interest else 0 for interest in unique_interests]

users_interest_vectors = [make_user_interest_vector(interest) for interest in users_interests]

user_similarities = [[cosine_sim(i,j) for j in users_interest_vectors ]for i in users_interest_vectors]

# Users 0 and 9 share interests in Hadoop, Java, and Big Data
assert 0.56 < user_similarities[0][9] < 0.58, "several shared interests"

# Users 0 and 8 share only one interest: Big Data
assert 0.18 < user_similarities[0][8] < 0.20, "only one shared interest"

In [7]:
def most_similar_user_to(user_id:int) -> List[Tuple[int, float]]:
    "wektory i id uzydkownikow ktorzy maja najwiecej wspolnych zainteresowan"
    return sorted([(id, val) for id, val in enumerate(user_similarities[user_id]) if id != user_id and val >0], key=lambda x: x[-1], reverse=True)
most_similar_user_to(0)

[(9, np.float64(0.5669467095138409)),
 (1, np.float64(0.33806170189140655)),
 (8, np.float64(0.1889822365046136)),
 (13, np.float64(0.16903085094570328)),
 (5, np.float64(0.1543033499620919))]

In [8]:
from typing import Dict
from collections import defaultdict
def user_based_suggestions(user_id: int,
                           include_current_interests: bool = False):
    # Sum up the similarities.
    suggestions: Dict[str, float] = defaultdict(float)
    for other_user_id, similarity in most_similar_user_to(user_id):
        for interest in users_interests[other_user_id]:
            suggestions[interest] += similarity

    
    suggestions = sorted(suggestions.items(),
                         key=lambda pair: pair[-1],  #
                         reverse=True)

    
    if include_current_interests:
        return suggestions
    else:
        return [(suggestion, weight)
                for suggestion, weight in suggestions
                if suggestion not in users_interests[user_id]]
    
user_based_suggestions(0)

[('MapReduce', np.float64(0.5669467095138409)),
 ('MongoDB', np.float64(0.5070925528371099)),
 ('Postgres', np.float64(0.5070925528371099)),
 ('NoSQL', np.float64(0.33806170189140655)),
 ('neural networks', np.float64(0.1889822365046136)),
 ('deep learning', np.float64(0.1889822365046136)),
 ('artificial intelligence', np.float64(0.1889822365046136)),
 ('databases', np.float64(0.16903085094570328)),
 ('MySQL', np.float64(0.16903085094570328)),
 ('Python', np.float64(0.1543033499620919)),
 ('R', np.float64(0.1543033499620919)),
 ('C++', np.float64(0.1543033499620919)),
 ('Haskell', np.float64(0.1543033499620919)),
 ('programming languages', np.float64(0.1543033499620919))]

In [9]:
# item based collaborative filtering

# wiersze to zainteresowania a kolumny to uzytkownicy
interest_user_matrix = [[user_interest_vector[j]
                         for user_interest_vector in users_interest_vectors] 
                        for j, _ in enumerate(unique_interests)]

interest_similarities = [[cosine_sim(user_vector_i, user_vector_j)
                          for user_vector_j in interest_user_matrix]
                         for user_vector_i in interest_user_matrix]

def most_similar_interests_to(interest_id: int):
    similarities = interest_similarities[interest_id]
    pairs = [(unique_interests[other_interest_id], similarity)
             for other_interest_id, similarity in enumerate(similarities)
             if interest_id != other_interest_id and similarity > 0]
    return sorted(pairs,
                  key=lambda pair: pair[-1],
                  reverse=True)
most_similar_interests_to(0)

[('Hadoop', np.float64(0.8164965809277259)),
 ('Java', np.float64(0.6666666666666667)),
 ('MapReduce', np.float64(0.5773502691896258)),
 ('Spark', np.float64(0.5773502691896258)),
 ('Storm', np.float64(0.5773502691896258)),
 ('Cassandra', np.float64(0.40824829046386296)),
 ('artificial intelligence', np.float64(0.40824829046386296)),
 ('deep learning', np.float64(0.40824829046386296)),
 ('neural networks', np.float64(0.40824829046386296)),
 ('HBase', np.float64(0.33333333333333337))]

In [10]:
def item_based_suggestions(user_id: int,
                           include_current_interests: bool = False):
    
    suggestions = defaultdict(float)
    user_interest_vector = users_interest_vectors[user_id]
    for interest_id, is_interested in enumerate(user_interest_vector):
        if is_interested == 1:
            similar_interests = most_similar_interests_to(interest_id)
            for interest, similarity in similar_interests:
                suggestions[interest] += similarity


    suggestions = sorted(suggestions.items(),
                         key=lambda pair: pair[-1],
                         reverse=True)

    if include_current_interests:
        return suggestions
    else:
        return [(suggestion, weight)
                for suggestion, weight in suggestions
                if suggestion not in users_interests[user_id]]
item_based_suggestions(0)

[('MapReduce', np.float64(1.861807319565799)),
 ('MongoDB', np.float64(1.3164965809277258)),
 ('Postgres', np.float64(1.3164965809277258)),
 ('NoSQL', np.float64(1.2844570503761732)),
 ('MySQL', np.float64(0.5773502691896258)),
 ('databases', np.float64(0.5773502691896258)),
 ('Haskell', np.float64(0.5773502691896258)),
 ('programming languages', np.float64(0.5773502691896258)),
 ('artificial intelligence', np.float64(0.40824829046386296)),
 ('deep learning', np.float64(0.40824829046386296)),
 ('neural networks', np.float64(0.40824829046386296)),
 ('C++', np.float64(0.40824829046386296)),
 ('Python', np.float64(0.2886751345948129)),
 ('R', np.float64(0.2886751345948129))]

In [11]:
#matrix factorization
from typing import NamedTuple

MOVIES = "ml-100k/u.item"
RATINGS = "ml-100k/u.data"

class Rating(NamedTuple):
    user_id: str
    movie_id: str
    rating: float


In [12]:
import csv

with open(MOVIES, encoding="iso-8859-1") as f:
    reader = csv.reader(f, delimiter="|")
    movies = {movie_id: title for movie_id, title, *_ in reader}


with open(RATINGS, encoding="iso-8859-1") as f:
    reader = csv.reader(f, delimiter="\t")
    ratings = [Rating(user_id, movie_id, float(rating))
                for user_id, movie_id, rating, _ in reader]


assert len(movies) == 1682
assert len(list({rating.user_id for rating in ratings})) == 943


In [13]:
import re

star_wars_ratings = {movie_id: []
                        for movie_id, title in movies.items()
                        if re.search("Star Wars|Empire Strikes|Return of the Jedi", title)}


for rating in ratings:
    if rating.movie_id in star_wars_ratings:
        star_wars_ratings[rating.movie_id].append(rating.rating)


avg_ratings = [(sum(title_ratings) / len(title_ratings), movie_id)
                for movie_id, title_ratings in star_wars_ratings.items()]


for avg_rating, movie_id in sorted(avg_ratings, reverse=True):
    print(f"{avg_rating:.2f} {movies[movie_id]}")


4.36 Star Wars (1977)
4.20 Empire Strikes Back, The (1980)
4.01 Return of the Jedi (1983)


In [14]:
import random
random.seed(0)
random.shuffle(ratings)

split1 = int(len(ratings) * 0.7)
split2 = int(len(ratings) * 0.85)

train = ratings[:split1]              
validation = ratings[split1:split2]   
test = ratings[split2:]

In [15]:
#baseline model
avg_rating = sum(rating.rating for rating in train) / len(train)
baseline_error = sum((rating.rating - avg_rating) ** 2
                        for rating in test) / len(test)


assert 1.26 < baseline_error < 1.27


In [16]:
import numpy as np
EMBEDDING_DIM = 2
Tensor = List
def random_tensor(*dims: int, init: str = 'normal') -> Tensor:
    if init == 'normal':
        return np.random.normal(size= dims)


user_ids = {rating.user_id for rating in ratings}
movie_ids = {rating.movie_id for rating in ratings}

user_vectors = {user_id: random_tensor(EMBEDDING_DIM)
                for user_id in user_ids}
movie_vectors = {movie_id: random_tensor(EMBEDDING_DIM)
                    for movie_id in movie_ids}

In [17]:
import tqdm
from typing import List

def loop(dataset: List[Rating], learning_rate: float = None) -> None:
    with tqdm.tqdm(dataset) as t:
        loss = 0.0
        for i, rating in enumerate(t):
            actual = rating.rating

            movie_vector = movie_vectors[rating.movie_id]
            user_vector = user_vectors[rating.user_id]            
            
            predicted = sum(u * v for u, v in zip(user_vector, movie_vector))
            error = predicted - actual
            loss += error ** 2
            
            if learning_rate:
            
                movie_grad = [2*error * u for u in user_vector]
                user_grad = [2*error * v for v in movie_vector]
                
            
                for j in range(EMBEDDING_DIM):
                    movie_vector[j] -= learning_rate * movie_grad[j]
                    user_vector[j] -= learning_rate * user_grad[j]
                    
            t.set_description(f'avg_loss: {loss / (i + 1):.4f}')

In [18]:
learning_rate = 0.05
for epoch in range(20):
    learning_rate *= 0.9
    print(epoch, learning_rate)
    loop(train, learning_rate=learning_rate)
    loop(validation)
loop(test)

0 0.045000000000000005


avg_loss: 1.3557: 100%|██████████| 15000/15000 [00:03<00:00, 3972.69it/s]


1 0.04050000000000001


avg_loss: 1.2546: 100%|██████████| 15000/15000 [00:03<00:00, 3978.30it/s]


2 0.03645000000000001


avg_loss: 1.2030: 100%|██████████| 15000/15000 [00:03<00:00, 3948.63it/s]


... (30 outputs left out)

18 0.006754258588364966


avg_loss: 0.9444: 100%|██████████| 15000/15000 [00:03<00:00, 3895.83it/s]


19 0.00607883272952847


avg_loss: 0.9225: 100%|██████████| 15000/15000 [00:03<00:00, 3921.80it/s]


In [19]:
def prediction(user_vectors, movie_vectors, test:Rating) -> Tuple[float, float]:
    user = test.user_id
    movie = test.movie_id
    rating = test.rating
    u = user_vectors[user]
    v = movie_vectors[movie]
    return rating, float(sum(u_i* v_i for u_i, v_i in zip(u,v)))
for i in range(0, len(test), 1000):
    print(prediction(user_vectors, movie_vectors, test[i]))


(4.0, 3.4432963388305757)
(3.0, 4.170621186399918)
(5.0, 4.424332303646153)
(4.0, 4.250545911766794)
(4.0, 3.35239858192587)
(2.0, 1.8486388665888795)
(1.0, 2.059706938766808)
(2.0, 2.6282853414635645)
(5.0, 3.7296706612113097)
(3.0, 2.9140413850012257)
(2.0, 2.7720987778713475)
(4.0, 3.5151830132597066)
(4.0, 3.7152222738523952)
(4.0, 3.7643256893274994)
(4.0, 3.4674543369292428)
